In [1]:
import stanfordnlp
stanfordnlp.download('en')

Using the default treebank "en_ewt" for language "en".
Would you like to download the models for: en_ewt now? (Y/n)

Default download directory: /Users/ayberkpalta/stanfordnlp_resources
Hit enter to continue or type an alternate directory.

Download location: /Users/ayberkpalta/stanfordnlp_resources/en_ewt_models.zip


100%|██████████| 235M/235M [04:18<00:00, 909kB/s] 



Download complete.  Models saved to: /Users/ayberkpalta/stanfordnlp_resources/en_ewt_models.zip
Extracting models file for: en_ewt
Cleaning up...Done.


In [2]:
from nltk.parse.corenlp import CoreNLPParser
from nltk.tokenize import word_tokenize

In [3]:
parser = CoreNLPParser(url='http://localhost:9000')
tokenizer = CoreNLPParser(url='http://localhost:9000', tagtype='pos')
pos_tagger = CoreNLPParser(url='http://localhost:9000', tagtype='pos')

In [4]:
sentence = "The quick brown fox jumps over the lazy dog."


In [5]:
tokens = word_tokenize(sentence)
print("Tokens:", tokens)

Tokens: ['The', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', '.']


In [7]:
pos_tags = list(pos_tagger.tag(tokens))
print("POS Tags:", pos_tags)

POS Tags: [('The', 'DT'), ('quick', 'JJ'), ('brown', 'JJ'), ('fox', 'NN'), ('jumps', 'VBZ'), ('over', 'IN'), ('the', 'DT'), ('lazy', 'JJ'), ('dog', 'NN'), ('.', '.')]


In [8]:
parse_tree = next(parser.raw_parse(sentence))
print("Parse Tree:")
print(parse_tree)

Parse Tree:
(ROOT
  (S
    (NP (DT The) (JJ quick) (JJ brown) (NN fox))
    (VP (VBZ jumps) (PP (IN over) (NP (DT the) (JJ lazy) (NN dog))))
    (. .)))


In [9]:
parse_tree.pretty_print()

                     ROOT                          
                      |                             
                      S                            
       _______________|__________________________   
      |                         VP               | 
      |                _________|___             |  
      |               |             PP           | 
      |               |     ________|___         |  
      NP              |    |            NP       | 
  ____|__________     |    |     _______|____    |  
 DT   JJ    JJ   NN  VBZ   IN   DT      JJ   NN  . 
 |    |     |    |    |    |    |       |    |   |  
The quick brown fox jumps over the     lazy dog  . 



LOCATION CHUNKER

In [10]:
from nltk.chunk import ChunkParserI
from nltk.chunk.util import conlltags2tree
from nltk.corpus import gazetteers

class LocationChunker(ChunkParserI):
    def __init__(self):
        self.locations = set(gazetteers.words())
        self.lookahead = 0
        for loc in self.locations:
            nwords = loc.count(' ')
        if nwords > self.lookahead:
            self.lookahead = nwords

In [12]:
def iob_locations(self, tagged_sent):
    
    i = 0
    l = len(tagged_sent)
    inside = False
    
    while i < l:
        word, tag = tagged_sent[i]
        j = i + 1
        k = j + self.lookahead
        nextwords, nexttags = [], []
        loc = False
        
    while j < k:
        if ' '.join([word] + nextwords) in self.locations:
            if inside:
                yield word, tag, 'I-LOCATION'
            else:
                yield word, tag, 'B-LOCATION'
            for nword, ntag in zip(nextwords, nexttags):
                yield nword, ntag, 'I-LOCATION'
                loc, inside = True, True
                i = j
                break
            
        if j < l:
            nextword, nexttag = tagged_sent[j]
            nextwords.append(nextword)
            nexttags.append(nexttag)
            j += 1
        else:
            break
        if not loc:
            inside = False
            i += 1
            yield word, tag, 'O'
            
    def parse(self, tagged_sent):
        iobs = self.iob_locations(tagged_sent)
        return conlltags2tree(iobs)

In [13]:
from nltk.chunk import ChunkParserI
from chunkers import sub_leaves
from chunkers import LocationChunker

t = loc.parse([('San', 'NNP'), ('Francisco', 'NNP'),
               ('CA', 'NNP'), ('is', 'BE'), ('cold', 'JJ'), 
               ('compared', 'VBD'), ('to', 'TO'), ('San', 'NNP'),
               ('Jose', 'NNP'), ('CA', 'NNP')])

print ("Location : \n", sub_leaves(t, 'LOCATION'))

ModuleNotFoundError: No module named 'chunkers'